# Tiền xử lý Celeb-DF v2 — Cache cross-dataset cho Nhóm A (Spatial crop)

**Mục tiêu:** trích khung hình + cắt khuôn mặt cho toàn bộ **518 video trong
`List_of_testing_videos.txt`** (danh sách test CHÍNH THỨC của Celeb-DF, dùng bởi hầu hết mọi
paper để báo cáo AUC Celeb-DF — kể cả UCF, SBI mà bạn đang so sánh), theo **ĐÚNG cùng 1 pipeline
tiền xử lý** (`MTCNN`, `FACE_MARGIN=1.3`, `IMG_SIZE=256`, `NUM_FRAMES=32` cách đều nhau) như
`prepareData_16072026_v1.ipynb` đã dùng cho FF++ — bắt buộc phải giống hệt để checkpoint đã train
trên FF++ nhận đúng phân bố dữ liệu khi eval zero-shot trên Celeb-DF.

**Dùng cho 4 model:** UCF, Xception, SBI, Spatial-only (Nhóm A theo phân loại đã thống nhất).
CAN/rPPG/Wu et al. cần cache khác (Nhóm B/C — làm ở bước sau).

**Đây là bước ĐÁNH GIÁ ZERO-SHOT** — không train/fine-tune gì trên Celeb-DF, nên **không cần
train/val split**, chỉ có đúng 1 tập `test`.

## ⚠️ An toàn nhãn 0/1 — ĐIỂM DỄ SAI NHẤT
`List_of_testing_videos.txt` có 1 quy ước nhãn từng gây nhầm lẫn ở nhiều paper/codebase khác
nhau (không thống nhất giữa các bản mô tả). Notebook này **KHÔNG tin vào số 0/1 ghi trong file**
— nhãn thật được **suy ra trực tiếp từ tên thư mục** trong đường dẫn (`Celeb-real`/`YouTube-real`
→ real=0, `Celeb-synthesis` → fake=1), đây là nguồn sự thật duy nhất không thể nhầm. Notebook sẽ
in ra bảng đối chiếu số nhãn trong file gốc vs nhãn suy từ thư mục — dừng lại và cảnh báo nếu có
bất kỳ dòng nào lệch nhau, để bạn tự kiểm tra trước khi chạy tiếp.

## 📋 Trước khi chạy
1. Add Input → dataset `reubensuju/celeb-df-v2` (hoặc dataset Celeb-DF v2 khác — sửa `CELEBDF_ROOT`).
2. Settings → Accelerator → GPU (cần cho MTCNN).
3. Settings → Internet → ON (tải pretrained MTCNN nếu chưa cache).


## 1. Cấu hình

In [ ]:
!pip install -q facenet-pytorch --no-deps


In [ ]:
import os, json, cv2, glob, time, gc, shutil
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import warnings
warnings.filterwarnings("ignore")

# ==================== ĐƯỜNG DẪN DATASET GỐC ====================
CELEBDF_ROOT = "/kaggle/input/celeb-df-v2"   # SỬA nếu tên dataset/đường dẫn khác
LIST_TEST_TXT = os.path.join(CELEBDF_ROOT, "List_of_testing_videos.txt")

assert os.path.isdir(CELEBDF_ROOT), f"Không tìm thấy CELEBDF_ROOT: {CELEBDF_ROOT}"
assert os.path.exists(LIST_TEST_TXT), (
    f"Không tìm thấy {LIST_TEST_TXT} -- kiểm tra lại CELEBDF_ROOT có đúng version chứa "
    f"List_of_testing_videos.txt hay không."
)

# ==================== NƠI LƯU CACHE ẢNH ĐÃ CROP ====================
OUT_DIR = "/kaggle/working/celebdf_v2_cache"
os.makedirs(OUT_DIR, exist_ok=True)

# ==================== THAM SỐ TRÍCH XUẤT — GIỐNG HỆT prepareData_16072026_v1.ipynb ====================
# BẮT BUỘC giống hệt cache FF++ để tương thích với checkpoint đã train (Mục 1 master-prompt-chuan-hoa-v2.md)
NUM_FRAMES   = 32
FACE_MARGIN  = 1.3
IMG_SIZE     = 256
JPEG_QUALITY = 92
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("CELEBDF_ROOT tồn tại:", os.path.isdir(CELEBDF_ROOT))
print("OUT_DIR:", OUT_DIR)
print(os.listdir(CELEBDF_ROOT))


## 2. Đọc `List_of_testing_videos.txt` — xác thực chéo nhãn qua tên thư mục

Không tin số 0/1 trong file — chỉ dùng file này để lấy DANH SÁCH ĐƯỜNG DẪN 518 video chính thức,
nhãn thật lấy từ tên thư mục cha trong path.

In [ ]:
FOLDER_TO_LABEL = {"Celeb-real": 0, "YouTube-real": 0, "Celeb-synthesis": 1}
FOLDER_TO_METHOD = {"Celeb-real": "original", "YouTube-real": "original", "Celeb-synthesis": "Celeb-synthesis"}

rows = []
n_mismatch = 0
with open(LIST_TEST_TXT, "r") as f:
    lines = [l.strip() for l in f.readlines() if l.strip()]

print(f"Tổng số dòng trong List_of_testing_videos.txt: {len(lines)}")

for line in lines:
    parts = line.split()
    assert len(parts) == 2, f"Dòng không đúng định dạng '<label> <path>': {line!r}"
    label_in_file, rel_path = parts
    label_in_file = int(label_in_file)

    folder = rel_path.split("/")[0]
    assert folder in FOLDER_TO_LABEL, f"Thư mục lạ không nhận diện được: {folder} (dòng: {line!r})"
    label_true = FOLDER_TO_LABEL[folder]   # <-- nguồn sự thật DUY NHẤT, không dùng label_in_file

    if label_in_file != label_true:
        n_mismatch += 1

    video_id = rel_path.replace("/", "__").rsplit(".", 1)[0]   # vd Celeb-synthesis__id0_id1_0000
    rows.append({
        "video_id": video_id, "split": "test", "label": label_true,
        "method": FOLDER_TO_METHOD[folder],
        "src_path": os.path.join(CELEBDF_ROOT, rel_path),
        "label_in_file_raw": label_in_file,
    })

manifest = pd.DataFrame(rows)
print(f"\nSố dòng nhãn-trong-file KHÁC nhãn-suy-từ-thư-mục: {n_mismatch} / {len(manifest)}")
print("(Nếu số này > 0, file txt dùng quy ước 0/1 khác chuẩn dự án -- nhưng KHÔNG ảnh hưởng gì vì "
      "ta luôn dùng label suy từ thư mục, không dùng số trong file.)")

print("\n=== Phân bố theo method (dùng nhãn ĐÃ XÁC THỰC qua thư mục) ===")
print(manifest.groupby(["label", "method"])["video_id"].count())

missing = manifest[~manifest["src_path"].apply(os.path.exists)]
print(f"\nSố video KHÔNG tìm thấy file trên đĩa: {len(missing)} / {len(manifest)}")
if len(missing) > 0:
    print(missing[["video_id", "src_path"]].head(10).to_string(index=False))
manifest = manifest[manifest["src_path"].apply(os.path.exists)].reset_index(drop=True)
print(f"\nSố video sẽ xử lý: {len(manifest)}")


## 3. Face detector (MTCNN) & hàm trích + cắt khuôn mặt — copy nguyên xi từ prepareData_16072026_v1.ipynb

In [ ]:
from facenet_pytorch import MTCNN

mtcnn = MTCNN(image_size=IMG_SIZE, margin=0, keep_all=False,
              post_process=False, select_largest=True, device=DEVICE)


def crop_face_frame(frame_rgb):
    """Phát hiện khuôn mặt + cắt theo margin; fallback center-crop nếu không detect được."""
    h, w = frame_rgb.shape[:2]
    try:
        boxes, probs = mtcnn.detect(frame_rgb)
    except Exception:
        boxes, probs = None, None

    if boxes is not None and len(boxes) > 0:
        idx = int(np.argmax(probs))
        x1, y1, x2, y2 = boxes[idx]
        bw, bh = x2 - x1, y2 - y1
        cx, cy = x1 + bw / 2, y1 + bh / 2
        half = max(bw, bh) * FACE_MARGIN / 2
        x1, y1 = int(max(cx - half, 0)), int(max(cy - half, 0))
        x2, y2 = int(min(cx + half, w)), int(min(cy + half, h))
    else:
        side = min(h, w)
        x1, y1 = (w - side) // 2, (h - side) // 2
        x2, y2 = x1 + side, y1 + side

    crop = frame_rgb[y1:y2, x1:x2]
    if crop.size == 0:
        side = min(h, w)
        x1, y1 = (w - side) // 2, (h - side) // 2
        crop = frame_rgb[y1:y1 + side, x1:x1 + side]
    return cv2.resize(crop, (IMG_SIZE, IMG_SIZE))


def extract_clip(video_path, out_dir, num_frames=NUM_FRAMES):
    """Trích num_frames khung hình cách đều nhau, cắt mặt, lưu JPEG."""
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return 0

    idxs = np.linspace(0, max(total - 1, 0), num_frames).astype(int)
    idx_set = set(idxs.tolist())
    saved, cur = 0, 0
    while cur <= idxs[-1]:
        ret, frame = cap.read()
        if not ret:
            break
        if cur in idx_set:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            crop = crop_face_frame(frame_rgb)
            out_path = os.path.join(out_dir, f"frame_{saved:02d}.jpg")
            cv2.imwrite(out_path, cv2.cvtColor(crop, cv2.COLOR_RGB2BGR),
                        [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
            saved += 1
        cur += 1
    cap.release()
    return saved


def print_disk_usage(path="/kaggle/working"):
    total, used, free = shutil.disk_usage(path)
    print(f"    [Disk {path}] Used: {used/1e9:.2f} GB | Free: {free/1e9:.2f} GB")


## 4. Vòng lặp xử lý chính — có checkpoint/resume

518 video, ít hơn nhiều so với FF++ (5000 video) nên chạy nhanh hơn đáng kể — không cần chia
nhóm theo split (chỉ có 1 split "test" duy nhất), nhưng vẫn giữ resume-safe để phòng sự cố.

In [ ]:
from tqdm import tqdm

CHECKPOINT_CSV = os.path.join(OUT_DIR, "manifest_processed.csv")

def process_all(df, out_dir_root):
    results = []
    for i, row in enumerate(tqdm(df.itertuples(), total=len(df))):
        sub = "real" if row.label == 0 else f"fake_{row.method}"
        out_dir = os.path.join(out_dir_root, row.split, sub, row.video_id)

        if os.path.isdir(out_dir) and len(glob.glob(os.path.join(out_dir, "*.jpg"))) >= NUM_FRAMES:
            results.append(dict(video_id=row.video_id, split=row.split, label=row.label,
                                 method=row.method, out_dir=out_dir, n_frames=NUM_FRAMES,
                                 status="skip_exists"))
            continue

        try:
            n = extract_clip(row.src_path, out_dir, NUM_FRAMES)
            status = "ok" if n >= NUM_FRAMES * 0.8 else "partial"
        except Exception as e:
            n, status = 0, f"error: {e}"

        results.append(dict(video_id=row.video_id, split=row.split, label=row.label,
                             method=row.method, out_dir=out_dir, n_frames=n, status=status))

        if i % 100 == 0 and i > 0:
            print_disk_usage()
            gc.collect()

    return pd.DataFrame(results)


if os.path.exists(CHECKPOINT_CSV):
    print("Tìm thấy checkpoint cũ, sẽ tiếp tục dựa trên các thư mục đã có sẵn.")

final_manifest = process_all(manifest, OUT_DIR)
final_manifest.to_csv(CHECKPOINT_CSV, index=False)

print("\n================ HOÀN TẤT TIỀN XỬ LÝ CELEB-DF V2 ================")
print(final_manifest["status"].value_counts())
print_disk_usage()


## 5. Kiểm tra kết quả — số liệu thật, ảnh mẫu

In [ ]:
print("=== Số video xử lý THÀNH CÔNG theo label (0=real, 1=fake) ===")
ok_mask = final_manifest["status"].isin(["ok", "skip_exists"])
print(final_manifest[ok_mask].groupby(["label", "method"])["video_id"].count())

n_fail = (~ok_mask).sum()
print(f"\nSố video LỖI/THIẾU khung hình: {n_fail} / {len(final_manifest)}")
if n_fail > 0:
    print(final_manifest[~ok_mask][["video_id", "status"]].head(20).to_string(index=False))

# Hiện vài ảnh mẫu để kiểm tra trực quan việc crop mặt có ổn không
import matplotlib.pyplot as plt
sample_dirs = final_manifest[ok_mask].sample(min(6, ok_mask.sum()), random_state=42)
fig, axes = plt.subplots(1, len(sample_dirs), figsize=(3 * len(sample_dirs), 3))
if len(sample_dirs) == 1: axes = [axes]
for ax, (_, row) in zip(axes, sample_dirs.iterrows()):
    imgs = sorted(glob.glob(os.path.join(row["out_dir"], "*.jpg")))
    if imgs:
        img = cv2.cvtColor(cv2.imread(imgs[len(imgs)//2]), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
    ax.set_title(f"{row['label']}/{row['method']}\n{row['video_id'][:20]}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "sample_crops_preview.png"), dpi=100)
plt.show()

total, used, free = shutil.disk_usage(OUT_DIR)
print(f"\nTổng dung lượng cache: {used/1e9:.2f} GB")


## ⚠️ 6. Lưu cache vĩnh viễn

Giống hệt hướng dẫn ở `prepareData_16072026_v1.ipynb` Mục 8:

1. **Save Version → Save & Run All (Commit)**.
2. Sau khi chạy xong, vào tab **Output** → **New Dataset** từ thư mục `/kaggle/working/celebdf_v2_cache/`.
3. Đặt tên dataset dễ nhớ, vd `celebdf-v2-face-cache`.
4. Dùng dataset này làm `CACHE_DIR` cho **4 notebook eval-only Celeb-DF** (UCF, Xception, SBI,
   Spatial-only) — mỗi notebook chỉ cần đọc `manifest_processed.csv` (toàn bộ đều `split="test"`)
   và load checkpoint FF++ đã train sẵn, KHÔNG train/fine-tune gì thêm.

**Lưu ý khác biệt với FF++:** Celeb-DF chỉ có **1 loại fake** (`method="Celeb-synthesis"`), nên
bước breakdown-theo-phương-pháp ở các notebook eval sẽ chỉ có 1 dòng (không có 4 dòng
Deepfakes/Face2Face/FaceSwap/NeuralTextures như FF++) — đây là điều bình thường, không phải lỗi.
